In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(
    Path(__file__).resolve().parent.parent
    / "data"
    / "raw"
    / "02_nav_history.csv"
)

print("Original Shape:", df.shape)

df["date"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

invalid_dates = df["date"].isna().sum()

print("\nInvalid Dates:", invalid_dates)

df = df.sort_values(
    ["amfi_code", "date"]
)

print("\nData sorted by amfi_code and date")

duplicates = df.duplicated().sum()

print("\nDuplicate Rows:", duplicates)

if duplicates > 0:
    print("Removing duplicate rows...")
    df = df.drop_duplicates()

missing_before = df["nav"].isna().sum()

print("\nMissing NAV Before Fill:", missing_before)

df["nav"] = (
    df.groupby("amfi_code")["nav"]
      .ffill()
)

missing_after = df["nav"].isna().sum()

print("Missing NAV After Fill:", missing_after)

invalid_nav = (df["nav"] <= 0).sum()

print("\nInvalid NAV Values:", invalid_nav)

if invalid_nav > 0:
    print("Removing invalid NAV rows...")
    df = df[df["nav"] > 0]

print("\nFinal Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

df.to_csv(
    "../data/processed/nav_history_clean.csv",
    index=False
)

In [ ]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(
    Path(__file__).resolve().parent.parent
    / "data"
    / "raw"
    / "07_scheme_performance.csv"
) 

print("Original Shape:", df.shape)

return_columns = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct"
]

for col in return_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    invalid_count = df[col].isna().sum()

    print(f"\nInvalid values in {col}:",
          invalid_count)

print("\nMissing Values:")
print(df.isnull().sum())

expense_anomalies = df[
    (df["expense_ratio_pct"] < 0.1)
    |
    (df["expense_ratio_pct"] > 2.5)
]

print(
    "\nExpense Ratio Anomalies:",
    len(expense_anomalies)
)

if len(expense_anomalies) > 0:

    print("\nAnomalous Records:")

    print(
        expense_anomalies[
            [
                "scheme_name",
                "expense_ratio_pct"
            ]
        ]
    )

return_anomalies = df[
    (df["return_1yr_pct"] < -100)
    |
    (df["return_1yr_pct"] > 100)
    |
    (df["return_3yr_pct"] < -100)
    |
    (df["return_3yr_pct"] > 100)
    |
    (df["return_5yr_pct"] < -100)
    |
    (df["return_5yr_pct"] > 100)
]

print(
    "\nReturn Value Anomalies:",
    len(return_anomalies)
)

if len(return_anomalies) > 0:

    print("\nAnomalous Return Records:")

    print(
        return_anomalies[
            [
                "scheme_name",
                "return_1yr_pct",
                "return_3yr_pct",
                "return_5yr_pct"
            ]
        ]
    )

duplicates = df.duplicated().sum()

print("\nDuplicate Rows:", duplicates)

if duplicates > 0:

    print("Removing duplicate rows...")

    df = df.drop_duplicates()

print("\nFinal Shape:", df.shape)

print("\nData Types:")
print(df.dtypes)


df.to_csv(
    "../data/processed/scheme_performance_clean.csv",
    index=False
)



In [ ]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(
    Path(__file__).resolve().parent.parent
    / "data"
    / "raw"
    / "08_investor_transactions.csv"
)


print("Original Shape:", df.shape)

df["transaction_date"] = pd.to_datetime(
    df["transaction_date"],
    errors="coerce"
)

# Check invalid dates
invalid_dates = df["transaction_date"].isna().sum()
print("Invalid Dates:", invalid_dates)

print("\nTransaction Types Before:")
print(df["transaction_type"].unique())

df["transaction_type"] = (
    df["transaction_type"]
    .astype(str)
    .str.strip()
    .str.lower()
)

mapping = {
    "sip": "SIP",
    "lumpsum": "Lumpsum",
    "redemption": "Redemption"
}

df["transaction_type"] = (
    df["transaction_type"]
    .replace(mapping)
)

print("\nTransaction Types After:")
print(df["transaction_type"].unique())

invalid_amounts = (df["amount_inr"] <= 0).sum()

print("\nInvalid Amounts:", invalid_amounts)

if invalid_amounts > 0:
    print("Removing invalid amount rows...")
    df = df[df["amount_inr"] > 0]

print("\nKYC Status Values:")
print(df["kyc_status"].unique())

allowed_kyc = [
    "Verified",
    "Pending"
]

invalid_kyc = df[
    ~df["kyc_status"].isin(allowed_kyc)
]

print("Invalid KYC Records:", len(invalid_kyc))

duplicates = df.duplicated().sum()

print("\nDuplicate Rows:", duplicates)

if duplicates > 0:
    df = df.drop_duplicates()

print("\nFinal Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())


df.to_csv(
    "../data/processed/investor_transactions_clean.csv",
    index=False
)